# Curriculum 05 · Lab 6 — Pseudo-relevance feedback: expand the query with retrieved terms

**Goal:** Query expansion with ZERO LLM calls. Every other lab in this track
pays for transformation with an LLM. PRF instead treats the retriever's own
first pass as feedback: stage 1 retrieves with the raw query, harvests the
top terms from the top-k documents it found (minus stopwords and terms
already in the query), appends them to the query, and retrieves again.

```
Transformer : PRFRetriever (tools/prf.py — a LangChain gap, hence tools/)
Flow        : raw query -> top-k feedback -> top terms -> expanded query -> top-k
Parameters  : feedback_k=3, n_terms=5, min_term_len=3, PRF_STOPWORDS
LLM         : none — pure Python (re + Counter), fully deterministic
Store       : FAISSVectorStore (in-memory) + SimilarityRetriever
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data        : rag-mini-wikipedia — first 100 passages, questions 1606/1610/1626
```

**Why PRF:** the expansion terms pull stage 2 toward the vocabulary of the
documents the first stage already judged relevant — no LLM, no training, no
index rebuild. The trade-off vs the LLM-based labs: terms come from
*retrieved* text, so PRF amplifies whatever the first pass found, good or
bad — garbage-in-garbage-out is stronger here.

This is the last lab of track 05-query-transformation (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present) and
puts the repo-root component library on `sys.path` so this notebook reuses
`src/tools/prf.py`, `src/retrieval/similarity.py`, `src/vectordb/faiss.py` and
`src/embeddings/bge.py` exactly like the lab script.

**WHY:** This is the one lab in the track with **no LLM anywhere** — no
`.env`, no API key, no Groq. Embeddings are local BGE and every other step
is pure Python, so the whole run is deterministic and fully offline.

**Paths:** the next cell resolves the **repo root** automatically and `cd`s
into it so every path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model loads lazily when
the experiment cell first calls it.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (embeddings/bge.py)
#   faiss-cpu             -> the FAISS index (vectordb/faiss.py)
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from embeddings.bge import BGEEmbedding  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from retrieval.similarity import SimilarityRetriever  # noqa: E402
from tools.prf import PRFRetriever  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The corpus constants (`N_PASSAGES = 100`, `QUESTION_IDS =
[1606, 1610, 1626]`) plus the PRF parameters: `TOP_K = 3` (stage-2 depth),
`FEEDBACK_K = 3` (stage-1 documents the terms are harvested from),
`N_TERMS = 5` (expansion terms added to the query), and the BGE model name.

**WHY:** `feedback_k < top_k` would be pointless (terms harvested from more
docs than returned) and `n_terms` is the expansion budget — the two knobs
together control how far stage 2 drifts from the raw query.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1626]  # same questions as labs 01/02/04/05, for comparison
TOP_K = 3  # stage-2 retrieval depth
FEEDBACK_K = 3  # how many stage-1 documents the expansion terms are harvested from
N_TERMS = 5  # how many terms the expanded query gains
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2 · Load — corpus + questions from the fresh parquet files

**WHAT:** `load_passages` pulls the first `n` passages (text + ids) from
`passages.parquet`; `load_questions` pulls specific rows by id from
`test.parquet`; `preview` flattens a passage for one-line printing.

**WHY:** Identical helpers to the rest of the track — same corpus, same
questions, one more transformation to compare side by side.


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3 · Experiment — raw vs PRF retrieval for the same questions

**WHAT:** `run_experiment` embeds the 100-passage subset once, builds the
FAISS store, then per question runs the plain `SimilarityRetriever` AND the
`PRFRetriever` over the same store — replaying PRF's two stages so the demo
can show the harvested terms and the expanded query, then recording the
stage-2 top-k and its wall time.

**WHY:** The stage replay is free (same three stage-1 retrievals the PRF
path already performs) and makes the expansion visible: terms -> expanded
query -> new top-k, all from one shared `exp`.


In [5]:
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory ---------------------------
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    store = FAISSVectorStore(embedding=embedder)
    t0 = time.perf_counter()
    store.add(chunks, embeddings=passage_vecs)
    index_s = time.perf_counter() - t0

    # --- The two retrievers over the SAME store -----------------------------
    raw_retriever = SimilarityRetriever(store, top_k=TOP_K)
    prf_retriever = PRFRetriever(
        raw_retriever, top_k=TOP_K, feedback_k=FEEDBACK_K, n_terms=N_TERMS
    )

    # --- Per question: raw retrieval + the PRF expansion + stage-2 retrieval -
    results = []
    for qid, qtext in questions:
        raw_docs = raw_retriever.retrieve(qtext)

        # Replay PRF's two stages so the demo can show what changed.
        feedback = raw_retriever.retrieve(qtext)[:FEEDBACK_K]
        terms = prf_retriever._feedback_terms(qtext, feedback, N_TERMS)
        expanded = " ".join([qtext, *terms]) if terms else qtext

        t0 = time.perf_counter()
        prf_docs = prf_retriever.retrieve(qtext)
        prf_s = time.perf_counter() - t0

        results.append(
            {
                "qid": qid,
                "question": qtext,
                "terms": terms,
                "expanded": expanded,
                "prf_s": prf_s,
                "raw_docs": raw_docs,
                "prf_docs": prf_docs,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "embed_s": embed_s,
        "index_s": index_s,
        "results": results,
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — one embed, one index build, all
retrievals (plus the Groq transformation calls) — and keeps the artifact
dict as `exp`.

**WHY:** Everything after this cell (the demo and the verification gate)
reads from this single `exp`, so the printed numbers and the verified
numbers are guaranteed to come from the same run. The LLM calls happen here,
once — the gate cell never re-burns them.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the corpus summary, then per question the
harvested feedback terms, the expanded query the stage-2 retriever actually
saw, and the top-1 of both the raw and the PRF path.

**WHY:** The expanded query is the whole lab — the raw question plus terms
the first stage found relevant. Compare the top-1s to see stage 2 pulled
toward the feedback documents' vocabulary; the terms list shows *what* moved
it there.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 06 — Pseudo-relevance feedback: expand the query with retrieved terms")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> PRF (no LLM)")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    embedded in {exp['embed_s']:.2f}s (dim 768), indexed in {exp['index_s']:.3f}s")

    print(f"\n[2] Raw vs PRF (per question):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        print(f"      feedback terms ({N_TERMS} max): {r['terms']}")
        print(f"      expanded query: {r['expanded']!r}")
        print(f"      raw  top-1: {preview(r['raw_docs'][0].page_content)}")
        print(f"      prf  top-1: {preview(r['prf_docs'][0].page_content)}")

    print("\n[3] Takeaway")
    print("    PRF is query expansion without an LLM: stage 1 retrieves with")
    print("    the raw query, stage 2 with the query plus the top terms of")
    print("    the stage-1 documents. The expansion terms move the second")
    print("    query toward the vocabulary of what the first pass already")
    print("    judged relevant — cheap, deterministic, and free of API calls.")
    print("    Trade-off vs the LLM-based labs: the terms come from retrieved")
    print("    text, so PRF amplifies whatever the first pass found, good or")
    print("    bad — garbage-in-garbage-out is stronger here.")


In [8]:
print_demo(exp)


Lab 06 — Pseudo-relevance feedback: expand the query with retrieved terms
BAAI/bge-base-en-v1.5 (local) -> FAISS top-3 -> PRF (no LLM)

[1] Corpus (deterministic subset, no randomness):
    100 passages (first 100 of 3200, ids 0..99)
    embedded in 15.53s (dim 768), indexed in 0.053s

[2] Raw vs PRF (per question):

    Q[1606] "Is Uruguay's capital Montevideo?"
      feedback terms (5 max): ['spanish', 'buenos', 'aires', 'british', 'early']
      expanded query: "Is Uruguay's capital Montevideo? spanish buenos aires british early"
      raw  top-1: Montevideo, Uruguay's capital.
      prf  top-1: Uruguay's capital, Montevideo, was founded by the Spanish in t...

    Q[1610] "Who founded Montevideo?"
      feedback terms (5 max): ['uruguay', 'capital', 'spanish', 'early', 'century']
      expanded query: 'Who founded Montevideo? uruguay capital spanish early century'
      raw  top-1: Uruguay's capital, Montevideo, was founded by the Spanish in t...
      prf  top-1: Uruguay's capital

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: exactly `N_PASSAGES` indexed, every
question returning `TOP_K` hits on both paths, every question harvesting >= 1
feedback term, no expansion term appearing in the raw question (expansion
must add information, not repeat it), and the content checks — the PRF top-3
must still carry the answer's keyword (montevideo / spanish / 1930).

**WHY:** `python 06-prf.py --verify` must print 12/12 PASS; this cell proves
the notebook reproduces the verified `.py` exactly. No LLM is involved, so
this gate is fully deterministic — every run prints the same 12 PASS.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (fully deterministic — no LLM anywhere).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))
    checks.append(("each question returns TOP_K raw hits",
                   all(len(r["raw_docs"]) == TOP_K for r in exp["results"])))
    checks.append(("each question returns TOP_K PRF hits",
                   all(len(r["prf_docs"]) == TOP_K for r in exp["results"])))

    # The expansion must actually add terms, and they must be new information
    # (not stopwords, not the question's own tokens).
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        checks.append((f"{tag} harvested >= 1 feedback term", len(r["terms"]) >= 1))
        checks.append((f"{tag} expansion terms are not in the raw question",
                       not any(t in r["question"].lower() for t in r["terms"])))

    # Content checks: the expanded query must still surface the answer's
    # keyword. Q1606 -> Montevideo; Q1610 -> the Spanish; Q1626 -> 1930.
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        joined = " ".join(d.page_content for d in r["prf_docs"]).lower()
        if r["qid"] == 1606:
            kw = "montevideo"
        elif r["qid"] == 1610:
            kw = "spanish"
        else:  # 1626
            kw = "1930"
        checks.append((f"{tag} PRF top-{TOP_K} retains '{kw}'", kw in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


In [10]:
verify_gate(exp)


verification gate:
  [PASS] exactly 100 passages indexed
  [PASS] each question returns TOP_K raw hits
  [PASS] each question returns TOP_K PRF hits
  [PASS] Q1606 harvested >= 1 feedback term
  [PASS] Q1606 expansion terms are not in the raw question
  [PASS] Q1610 harvested >= 1 feedback term
  [PASS] Q1610 expansion terms are not in the raw question
  [PASS] Q1626 harvested >= 1 feedback term
  [PASS] Q1626 expansion terms are not in the raw question
  [PASS] Q1606 PRF top-3 retains 'montevideo'
  [PASS] Q1610 PRF top-3 retains 'spanish'
  [PASS] Q1626 PRF top-3 retains '1930'


0